# Offline Pipeline: Xây dựng Hệ thống Indexing (BM25 + FAISS)
Notebook này thực hiện bước chuẩn bị dữ liệu Offline.
1. Gộp tất cả các file `corpus_part_*.parquet` thành một Dataset thống nhất.
2. Cắt Chunk dữ liệu (sử dụng ContextualChunker hoặc SemanticChunker).
3. Tokenize và xây dựng BM25 Index.
4. Encode và xây dựng FAISS Index bằng model Qwen 4-bit.
5. Lưu toàn bộ kết quả ra ổ cứng để phục vụ cho bước Inference.


In [ ]:
# Cài đặt thư viện
!pip install -q pandas pyarrow rank_bm25 sentence-transformers faiss-cpu faiss-gpu bitsandbytes accelerate


In [ ]:
import pandas as pd
import glob
import os
import pickle
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch
import warnings
warnings.filterwarnings('ignore')

# 1. Đọc và gộp toàn bộ các file corpus parquet
corpus_files = glob.glob('/kaggle/working/corpus_text/corpus_part_*.parquet') # Đổi đường dẫn theo Kaggle của bạn
print(f"Tìm thấy {len(corpus_files)} file corpus.")

df_list = []
for f in corpus_files:
    df_list.append(pd.read_parquet(f))

if df_list:
    df_corpus = pd.concat(df_list, ignore_index=True)
    # Xóa các dòng rỗng (do lỗi hoặc timeout)
    df_corpus = df_corpus.dropna(subset=['text'])
    print(f"Tổng số URL thu thập thành công: {len(df_corpus)}")
else:
    print("Không tìm thấy file corpus nào! Chạy fake data để test code...")
    df_corpus = pd.DataFrame([
        {'id': 'doc_1', 'url': 'http://abc', 'text': 'Sốt xuất huyết là bệnh truyền nhiễm cấp tính.'},
        {'id': 'doc_2', 'url': 'http://def', 'text': 'Ung thư phổi là một trong những nguyên nhân tử vong hàng đầu.'}
    ])


In [ ]:
# Đưa code Chunker vào đây
# (Copy nguyên class BaseChunker, SentenceChunker, ContextualChunker, SemanticEmbeddingChunker, Chunker từ src/preprocessing/chunker.py)
import re
from typing import List, Dict, Any

class BaseChunker:
    def __init__(self, max_tokens: int = 1024, token_overlap: int = 100):
        self.max_tokens = max_tokens
        self.token_overlap = token_overlap
        self.max_words = int(self.max_tokens * 0.75)
        self.overlap_words = int(self.token_overlap * 0.75)
    def count_words(self, text: str) -> int: return len(text.split())
    def split_into_sentences(self, text: str) -> List[str]:
        sentences = re.split(r'(?<=[.!?。！？])\s+', text.strip())
        return [s.strip() for s in sentences if s.strip()]

class SentenceChunker(BaseChunker):
    def chunk_document(self, doc_id: str, text: str, url: str = "") -> List[Dict[str, Any]]:
        if not text: return []
        paragraphs = re.split(r'\n\s*\n', text.strip())
        chunks = []
        current_chunk_words = []
        current_word_count = 0
        def add_chunk(words_list):
            chunk_text = ' '.join(words_list).strip()
            if chunk_text:
                chunks.append({'doc_id': doc_id, 'chunk_index': len(chunks), 'content': chunk_text, 'url': url})
        for paragraph in paragraphs:
            para_words = paragraph.split()
            para_word_count = len(para_words)
            if current_word_count + para_word_count <= self.max_words:
                current_chunk_words.extend(para_words)
                current_word_count += para_word_count
            else:
                for sentence in self.split_into_sentences(paragraph):
                    sentence_words = sentence.split()
                    sentence_word_count = len(sentence_words)
                    if current_word_count + sentence_word_count > self.max_words and current_chunk_words:
                        add_chunk(current_chunk_words)
                        overlap_start = max(0, len(current_chunk_words) - self.overlap_words)
                        current_chunk_words = current_chunk_words[overlap_start:]
                        current_word_count = len(current_chunk_words)
                    current_chunk_words.extend(sentence_words)
                    current_word_count += sentence_word_count
        if current_chunk_words: add_chunk(current_chunk_words)
        return chunks

class ContextualChunker(SentenceChunker):
    def chunk_document(self, doc_id: str, text: str, url: str = "") -> List[Dict[str, Any]]:
        if not text: return []
        paragraphs = re.split(r'\n\s*\n', text.strip())
        global_context = ""
        for p in paragraphs:
            if len(p.split()) > 5:  
                global_context = ' '.join(p.split()[:40])
                break
        chunks = super().chunk_document(doc_id, text, url)
        for chunk in chunks:
            chunk_text = chunk['content']
            if global_context and global_context not in chunk_text:
                chunk['content'] = f"[Context: {global_context}] {chunk_text}"
        return chunks

class Chunker:
    def __init__(self, strategy: str = "contextual", **kwargs):
        strategies = {"sentence": SentenceChunker, "contextual": ContextualChunker}
        self.chunker = strategies[strategy](**kwargs)
    def chunk_document(self, doc_id: str, text: str, url: str = "") -> List[Dict[str, Any]]:
        return self.chunker.chunk_document(doc_id, text, url)


In [ ]:
# 2. Chunking dữ liệu
print("Bắt đầu Chunking toàn bộ văn bản...")
chunker = Chunker(strategy="contextual", max_tokens=512, token_overlap=50)

all_chunks = []
for idx, row in df_corpus.iterrows():
    doc_id = row['id']
    text = row['text']
    chunks = chunker.chunk_document(doc_id, text)
    all_chunks.extend(chunks)

df_chunks = pd.DataFrame(all_chunks)
print(f"Tổng số chunks tạo ra: {len(df_chunks)}")

# Lưu metadata của chunk (dùng để map kết quả trả về doc_id)
df_chunks.to_parquet('chunk_metadata.parquet', engine='pyarrow')


In [ ]:
# 3. Xây dựng BM25 Index
print("Bắt đầu Tokenize và xây dựng BM25 Index...")
# Tách từ đơn giản bằng khoảng trắng (có thể thay bằng Underthesea/VnCoreNLP nếu muốn)
tokenized_corpus = [doc.lower().split(" ") for doc in df_chunks['content'].tolist()]

bm25 = BM25Okapi(tokenized_corpus)

# Lưu BM25 ra ổ cứng
with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print("Đã lưu bm25_index.pkl")


In [ ]:
# 4. Xây dựng FAISS (Semantic) Index với Qwen 4-bit
print("Khởi tạo Qwen Embedding Model (4-bit)...")
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Chạy trên GPU
model_kwargs = {'device': 'cuda', 'quantization_config': quantization_config}
encode_kwargs = {'normalize_embeddings': True}

eval_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B", 
    model_kwargs=model_kwargs,
    trust_remote_code=True
)
eval_model.max_seq_length = 512

print("Bắt đầu tạo Embeddings cho toàn bộ Chunks...")
# Mã hóa toàn bộ
corpus_texts = [str(x)[:2000] for x in df_chunks['content'].tolist()]
corpus_embeddings = eval_model.encode(corpus_texts, batch_size=16, show_progress_bar=True)

print("Đang tạo FAISS Index...")
dimension = corpus_embeddings.shape[1]
# Dùng faiss.IndexFlatIP vì embeddings đã được normalize (tương đương Cosine Similarity)
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(corpus_embeddings)

# Lưu FAISS Index ra ổ cứng
faiss.write_index(faiss_index, 'faiss_index.bin')
print("Đã lưu faiss_index.bin")
print("🎉 HOÀN TẤT OFFLINE PIPELINE!")
